<h1 style="text-align: center; font-family: 'menlo'; color: #ADD8E6; font-size:50px;">
  <span style="background-color: #191970; padding: 5px 10px; border-radius: 5px; display: inline-block;">
         Reading images
  </span>
</h1>

Astropy provides a few ways to read in FITS images, some in the core package and others in affiliated packages.

Before exploring those, we’ll create a set of (fake) images to work with.

In [1]:
from pathlib import Path #manipular caminhos de pastas/arquivos
from astropy.nddata import CCDData #ler imagens FITS como objetos de CCD 
from astropy.io import fits #módulo básico do astropy pra abrir arquivos FITS

# Working with directories

The cell below contains the path to the images. In this notebook we’ll use it both to store the fake images we generate and to read images. In normal use, you wouldn’t start by writing images there, however.

If the images are in the same directory as the notebook, you can omit this or set it to an empty string ''. Having images in the same directory as the notebook is less complicated, but it’s not at all uncommon to need to work with images in a different directory.

Later, we’ll look at how to generate the full path to an image (directory plus file name) in a way that will work on any platform. One of the approaches to loading images (using <span style="color: #d62495">ccdproc.ImageFileCollection</span>) lets you mostly forget about this.

In [2]:
data_directory = 'path/to/my/images'

# Generate some fake images

The cells below generate some fake images to use later in the notebook.

In [3]:
from pathlib import Path
from itertools import cycle #importa a função cycle, que cria um iterador que repete infinitamente uma sequência
import numpy as np


data_directory = 'fake_imagesss' #pasta dentro do diretório atual do notebook, p funcionar a célula da frente
image_path = Path(data_directory) #converte data_directory em um objeto Path, permitindo manipular melhor o caminho
image_path.mkdir(parents=True, exist_ok=True) #cria o diretório onde as imagens serão salvas,, parents = cria os diretórios pais se não existirem,, ok = não dá erro se o diretório já existir

images_to_generate = {'BIAS': 5,
                     'DARK': 10,
                     'FLAT':3,
                     'LIGHT': 10}

exposure_times = {'BIAS': [0.0],
                 'DARK': [5.0, 30.0],
                 'FLAT': [5.0, 6.1, 7.3],
                 'LIGHT': [30.0]}
#tempo de exposição em segundos

filters = {'FLAT': 'V', 'LIGHT': 'V'}
#filtro V da banda visial, só esses dois tem

objects = {'LIGHT': ['m82', 'xx cyg']}
#só imagens light tem objeto associado, m82 é uma galáxia e xxcyg é uma estrela

image_size = [300, 200]
image_number = 0

for image_type, num in images_to_generate.items():
    exposure = cycle(exposure_times[image_type]) #lista de tempo de exposição pra iterar infinitamente, DARK: 5.0, 30.0, 5.0, 30.0, 5.0... 
    try:
        filts = cycle(filters[image_type]) #ciclo de filtros para esse tipo de imagem
    except KeyError:
        filts = [] #se o tipo nao tiver no dicionario filters (bias e dark), captura o keyerror e define filts como lista vazia

    try: 
        objs = cycle(objects[image_type])
    except KeyError:
        objs = [] #mesma coisa que acima mas com objetos
    for _ in range(num): #loop interno, repete num vezes (quantidade definida no images_to_generate)
        img = CCDData(data=np.random.randn(*image_size), unit='adu') #criação da imagem * desempacota [300, 200] para np.random.randn
        img.meta['IMAGETYP'] = image_type #adicionando os metadados, aqui é header FITS a palavra-chave IMAGETYP com o tipo da imagem (BIAS, DARK, FLAT ou LIGHT)
        img.meta['EXPOSURE'] = next(exposure) #Pega o próximo valor do ciclo de tempos de exposição e coloca no header como EXPOSURE.
        if filts:
            img.meta['FILTER'] = next(filts) #SE houver filtros para esse tipo, pega o próximo filtro do ciclo e coloca no header como FILTER
        if objs:
            img.meta['OBJECT'] = next(objs) #mesma coisa que acima mas SE houver objetos definidos
        image_name = str(image_path / f'img-{image_number:04d}.fits') #montando o nome do arquivo para salvar
        #image_path junta o diretório com o nome do arquivo (qualquer sistema operacional)
        #f'img formata o número com 4 digitos (0000, 0001, 0002...)
        #.fits pra ficar em formato FITS
        img.write(image_name, overwrite=True) #salva imagem no disco no formato FITS
        print(image_name)
        image_number += 1 #imprime o nome do arquivo gerado e imcrementa o contado para o próximo arquivo

fake_imagesss\img-0000.fits
fake_imagesss\img-0001.fits
fake_imagesss\img-0002.fits
fake_imagesss\img-0003.fits
fake_imagesss\img-0004.fits
fake_imagesss\img-0005.fits
fake_imagesss\img-0006.fits
fake_imagesss\img-0007.fits
fake_imagesss\img-0008.fits
fake_imagesss\img-0009.fits
fake_imagesss\img-0010.fits
fake_imagesss\img-0011.fits
fake_imagesss\img-0012.fits
fake_imagesss\img-0013.fits
fake_imagesss\img-0014.fits
fake_imagesss\img-0015.fits
fake_imagesss\img-0016.fits
fake_imagesss\img-0017.fits
fake_imagesss\img-0018.fits
fake_imagesss\img-0019.fits
fake_imagesss\img-0020.fits
fake_imagesss\img-0021.fits
fake_imagesss\img-0022.fits
fake_imagesss\img-0023.fits
fake_imagesss\img-0024.fits
fake_imagesss\img-0025.fits
fake_imagesss\img-0026.fits
fake_imagesss\img-0027.fits


# Option 1: Reading a single image with <span style="color: #d62495">astropy.io.fits</span>

This option gives you the most flexibility but is the <U>least adapted to CCD images</U> specifically. What you read in is a list of FITS extensions; you must first select the one you want then access the data or header as desired.

We’ll open up the first of the fake images, <span style="color: #d62495">img-0001.fits</span>. To combine that with the directory name we’ll use Python 3’s <span style="color: #d62495">pathlib</span>, which ensures that the path combination will work on Windows too.

In [4]:
image_name = 'img-0001.fits' #nome do arquivo antes de abrir
image_path_read = Path(data_directory) / image_name #cria um objeto Path com o caminho da pasta
                                                    # o operador / do pathlib junta o caminho da pasta com o nome do arquivo.

hdu_list = fits.open(image_path_read) #fits vem do astropy.io import fits,,, fits.open() abre o arquivo FITS e retorna um objeto do tipo HDUList.
hdu_list.info() #imprime um resumo de todos os HDUs dentro do arquivo. O output é tipo uma tabelinha.

Filename: fake_imagesss\img-0001.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       8   (200, 300)   float64   


The <span style="color: #d62495">hdu_list</span> is a list of FITS Header-Data Units. In this case there is just one, containing both the image header and data, which can be accessed as shown below.

In [5]:
hdu = hdu_list[0]
#arquivo .fits é como uma caixa que pode conter várias "folhas" (HDUs). Cada folha tem um header (metadados) e um data (array de pixels).
hdu.header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  200                                                  
NAXIS2  =                  300                                                  
IMAGETYP= 'BIAS    '                                                            
EXPOSURE=                  0.0                                                  
BUNIT   = 'adu     '                                                            

In [6]:
import pandas as pd

hduList = {
    'Name': ['SIMPLE', 'BITPIX', 'NAXIS', 'NAXIS1', 'NAXIS2', 'IMAGETYP', 'EXPOSURE', 'BUNIT'],
    'Output': ['T', -64, 2, 200, 300, 'BIAS', 0.0, 'adu' ],
    'Meaning': ['True, só diz que o arquivo segue o padrão FITS. Sempre é T.',  
                'Float64 (números decimais de 64 bits). Positivo = inteiro.', 
                'A imagem tem 2 dimensões (2D)',
                'Largura da imagem em pixels (eixo X, colunas)', 
                'Altura da imagem em pixels (eixo Y, linhas)',
                'Tipo da imagem. Pode ser BIAS, DARK, FLAT ou LIGHT (science).',
                'Tempo de exposição em segundos. BIAS = 0s (obturador fechado, só lê o detector).',
                'Unidade dos dados: ADU (contagens do detector).']
}

tabela = pd.DataFrame(hduList)

tabela.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

Name,Output,Meaning
SIMPLE,T,"True, só diz que o arquivo segue o padrão FITS. Sempre é T."
BITPIX,-64,Float64 (números decimais de 64 bits). Positivo = inteiro.
NAXIS,2,A imagem tem 2 dimensões (2D)
NAXIS1,200,"Largura da imagem em pixels (eixo X, colunas)"
NAXIS2,300,"Altura da imagem em pixels (eixo Y, linhas)"
IMAGETYP,BIAS,"Tipo da imagem. Pode ser BIAS, DARK, FLAT ou LIGHT (science)."
EXPOSURE,0.000000,"Tempo de exposição em segundos. BIAS = 0s (obturador fechado, só lê o detector)."
BUNIT,adu,Unidade dos dados: ADU (contagens do detector).


In [7]:
hdu.data
#array numpy = shape=(300, 200), dtype='float64'
#cada número é o valor de um pixel. No nosso caso são aleatórios (ruído), mas numa imagem real seriam as contagens de luz que o CCD capturou.

array([[ 0.24013982, -1.70306209,  0.24446763, ..., -0.61968012,
        -1.24233677,  1.04908186],
       [-0.30552463,  0.67679135,  0.76895754, ..., -0.6994783 ,
        -0.17890027, -0.90108751],
       [ 0.07950571,  2.07940702,  0.14714859, ..., -0.84770958,
        -0.30662508,  1.8780829 ],
       ...,
       [-0.16016164,  0.80561299, -0.95910687, ...,  0.8184773 ,
        -0.66825336,  1.07916562],
       [ 1.25286261, -0.69456848,  0.12288738, ..., -0.40440435,
         1.89931652,  2.50795771],
       [ 0.60008444, -0.0535546 ,  0.36196304, ...,  1.76875787,
        -0.65129395,  0.84195207]], shape=(300, 200), dtype='>f8')

The [documentation for io.fits](https://docs.astropy.org/en/stable/io/fits/index.html) describes more of its capabilities.

# Option 2: Use <span style="color: #d62495">CCDData</span> to read in a single image

Astropy contains a <span style="color: #d62495">CCDData</span> object for representing a single image. It’s not as flexible as using <span style="color: #d62495">astrop.io.fits</span> directly (for example, it assumes there is only one FITS extension and that it contains image data) but it sets up several properties that make the data easier to work with.

We’ll read in the same single image we did in the example above, <span style="color: #d62495">img-0001.fits.</span>.

In [15]:
data_directory = 'fake_imagesss'
image_name = 'img-0001.fits'
image_path = Path(data_directory) / image_name

ccd = CCDData.read(image_path) #esse sozinho n funciona pk eh a PASTA, tem que ESPECIFICAR o arquivo como eu fiz ai em cima

The data and header are accessed similarly to how you access it in an HDU returned by <span style="color: #d62495">astrop.io.fits</span>:

In [12]:
ccd.header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  200                                                  
NAXIS2  =                  300                                                  
IMAGETYP= 'BIAS    '                                                            
EXPOSURE=                  0.0                                                  
BUNIT   = 'adu     '                                                            

In [14]:
ccd.data

array([[ 0.24013982, -1.70306209,  0.24446763, ..., -0.61968012,
        -1.24233677,  1.04908186],
       [-0.30552463,  0.67679135,  0.76895754, ..., -0.6994783 ,
        -0.17890027, -0.90108751],
       [ 0.07950571,  2.07940702,  0.14714859, ..., -0.84770958,
        -0.30662508,  1.8780829 ],
       ...,
       [-0.16016164,  0.80561299, -0.95910687, ...,  0.8184773 ,
        -0.66825336,  1.07916562],
       [ 1.25286261, -0.69456848,  0.12288738, ..., -0.40440435,
         1.89931652,  2.50795771],
       [ 0.60008444, -0.0535546 ,  0.36196304, ...,  1.76875787,
        -0.65129395,  0.84195207]], shape=(300, 200), dtype='>f8')

There are a [number of features of CCDData](https://docs.astropy.org/en/stable/nddata/ccddata.html) that make it convenient for working with WCS, slicing, and more. Some of those features will be discussed in more detail in the notebooks that follow.

# Option 3: Working with a directory of images using <span style="color: #d62495">ImageFileCollection</span>

The affiliated package [ccdproc](https://ccdproc.readthedocs.io/en/latest/) provides an easier way to work with collections of images in a directory: an <span style="color: #d62495">ImageFileCollection</span>. The <span style="color: #d62495">ImageFileCollection</span> is initialized with the name of the directory containing the images.

In [22]:
from ccdproc import ImageFileCollection
im_collection = ImageFileCollection(data_directory)

# escaneia todos os arquivos FITS da pasta e lê os headers
# não carrega os dados das imagens na memória ainda, só os metadados (headers)

Note that we didn’t need to worry about using <span style="color: #d62495">pathlib</span> to combine the directory and file name, instead we give the collection the name of the directory.

## Summary of directory contents

The <span style="color: #d62495">summary</span> property provides an overview of the files in the directory: it’s an astropy <span style="color: #d62495">Table</span>, so you can access columns in the usual way.

In [17]:
im_collection.summary

file,simple,bitpix,naxis,naxis1,naxis2,imagetyp,exposure,bunit,filter,object
str13,bool,int64,int64,int64,int64,str5,float64,str3,object,object
img-0000.fits,True,-64,2,200,300,BIAS,0.0,adu,--,--
img-0001.fits,True,-64,2,200,300,BIAS,0.0,adu,--,--
img-0002.fits,True,-64,2,200,300,BIAS,0.0,adu,--,--
img-0003.fits,True,-64,2,200,300,BIAS,0.0,adu,--,--
img-0004.fits,True,-64,2,200,300,BIAS,0.0,adu,--,--
img-0005.fits,True,-64,2,200,300,DARK,5.0,adu,--,--
img-0006.fits,True,-64,2,200,300,DARK,30.0,adu,--,--
img-0007.fits,True,-64,2,200,300,DARK,5.0,adu,--,--
img-0008.fits,True,-64,2,200,300,DARK,30.0,adu,--,--


## Filtering and iterating over images

The great thing about <span style="color: #d62495">ImageFileCollection</span> is that it provides <u>convenient ways to filter or loop</u> over files via FITS header keyword values.

For example, looping over just the flat files is one line of code:

In [21]:
for a_flat in im_collection.hdus(imagetyp='FLAT'):
    print(a_flat.header['EXPOSURE'])

# pegou só os arquivos que tem FLAT, eram 3 e DENTRO de flat --> quais que tinham EXPOSURE
# FLAT é o filtro (qual tipo de arquivo quero que abra)
# EXPOSURE é o dado que eu quero extrair de dentro de cada arquivo

5.0
6.1
7.3


Instead of iterating over HDUs, as in the example above, you can iterate over just the headers (with  <span style="color: #d62495">.headers</span>) or just the data (with <span style="color: #d62495">.data</span>). You can use any FITS keyword from the header as a keyword for selecting the images you want. In addition, you can return the file name while also iterating.

*Iterar seria para conseguir trabalhar com os dados de MUITAS imagens. <br> Por exemplo: combinar vários bias para fazer um master bias, nesse exemplo se tirou 5 imagens de BIAS. Cada uma tem um pouco de ruído diferente. Pra eliminar o ruído, soma as 5 e divide por 5 (média). Sem loop, teria que abrir um por um dos arquivos:* <br> `bias1 = CCDData.read('img-0000.fits')` <br> `bias2 = CCDData.read('img-0001.fits')`

*Com ImageFileCollection:* <br> `for bias in im_collection.ccds(imagetyp='BIAS'):` <br> *Isso faz a média de todas automaticamente.* 

*Com todas as médias é possível fazer uma imagem calibrada como apresentado nos notebooks anteriores. Exemplo:* <br> `for light in im_collection.ccds(imagetyp='LIGHT'):` <br> `calibrada = (light - master_bias - master_dark) / master_flat` <br> `calibrada.write(f'calibrada_{light.header["OBJECT"]}.fits')`

In [23]:
for a_flat, fname in im_collection.hdus(imagetyp='LIGHT', object='m82', return_fname=True):
    print(f'In file {fname} the exposure is:', a_flat.header['EXPOSURE'], 'with a standard deviation', a_flat.data.std())

# imagetyp=LIGHT --> filtra só as science
# object=m82 --> filtra só as do objeto
# return_fname=True --> loop retorna dois valores: HDU e o nome do arquivo
# fname --> recebe o nome do arquivo (img-0005.fits)
# a_flat.data.std() --> calcula o desvio padrão dos pixels dessa imagem

In file img-0018.fits the exposure is: 30.0 with a standard deviation 0.9970746168482367
In file img-0020.fits the exposure is: 30.0 with a standard deviation 0.9998960338370341
In file img-0022.fits the exposure is: 30.0 with a standard deviation 0.9987020304297226
In file img-0024.fits the exposure is: 30.0 with a standard deviation 0.9988942636271465
In file img-0026.fits the exposure is: 30.0 with a standard deviation 0.9989617555983968


The [documentation for ImageFileCollection](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.ImageFileCollection.html) describes more of its capabilities. <span style="color: #d62495">ImageFileCollection</span> can automatically save a copy of each image as you iterate over them, for example.

In [24]:
for a_flat, fname in im_collection.ccds(bunit='ADU', return_fname=True):
    print(a_flat.unit)

adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu
adu


In [26]:
a_flat.header #a_flat eh um nome qualquer que foi escolhido, pode ser qualquer outro

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  200                                                  
NAXIS2  =                  300                                                  
IMAGETYP= 'LIGHT   '                                                            
EXPOSURE=                 30.0                                                  
FILTER  = 'V       '                                                            
OBJECT  = 'xx cyg  '                                                            
BUNIT   = 'adu     '                                                            

## Summary

Best use when

- Option 1: fits.open
    - sfsdf
---
- Options 2: CCDData.read
    - fsdf
---
- Option 3:
    - sdfsd
---

| czxc | Option 1: fits.open | Option 2: CCDData.read | Option 3: |
| :--- | :---: | :---: | :---: |
| What it is | Generic FITS file reader. Low-level, handles any FITS file. | Specialized CCD image object. High-level, astro-optimized. | OPISAO 3 |
| Returns | An HDUList — a list of Header-Data Units. | A single CCDData object with extras built-in. | OPISAO 3 |
| Acess Header | hdu_list[0].header | ccd.header | OPSAO 3 |
| Acess Data | hdu_list[0].data | ccd.data | OPSAO 3
| Units | Not tracked. Remember BUNIT manually. | Built-in (ccd.unit) | OPSAO 3
| Multi-extension | Handles multiple HDUs | Assumes only 1 extension with image data. | OPSAO 3
| WCS | Raw header only. Build WCS by yourself | ccd.wsc, ready to convert pixels | OPSAO 3
| Slicing | Loses WCS and metadata | Preserves WCS and header | OPSAO 3
| Best for | Complex files, spectra, tables, multi-HDU data. | Daily CCD reduction: bias, dark, flat, science images. | OPSAO 3


Alinhamento: Use dois pontos : na linha dos hífens para alinhar o texto:
:--- alinha à esquerda.
:---: centraliza.
---: alinha à direita.

<h1 style="text-align: left; font-family: 'menlo'; color: #191970; font-size:40px;">
  <span style="background-color: #ADD8E6; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Notes
  </span>
</h1>

- Matplotlib [axvline](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html) & [pyplot](https://matplotlib.org/stable/tutorials/pyplot.html).

- Usar Path em vez de juntar string: no Windows o separador é \ e no Linux/Mac é /. O pathlib cuida disso sozinho — seu código funciona em qualquer sistema operacional.  *Exemplo*: se data_directory = 'fake_images', o resultado é fake_images\img-0001.fits (no Windows) ou fake_images/img-0001.fits (no Linux)

- HDU = Header Data Units, é um arquivo FITS pode conter várias "extensões". Cada extensão tem:
    - Um header (metadados — palavras-chave como EXPOSURE, IMAGETYP, etc.)
    - Um data array (os pixels da imagem em si)

- WCS = World Coordinate System =  sistema que converte pixels em coordenadas do céu

- Lembrar que:
    - Iterar sobre imagens = automatizar tarefas repetitivas que você faria em dezenas de arquivos (ali na opção 3, ao invés de abrir um a um faz isso !!!)

---

<h1 style="text-align: left; font-family: 'menlo'; color: #191970; font-size:40px;">
  <span style="background-color: #ADD8E6; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Links
  </span>
</h1>

https://github.com/astropy/ccd-reduction-and-photometry-guide<br>https://github.com/nyny2903/astropy-coisas/tree/main<br>https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/01-11-reading-images.html

https://matplotlib.org/stable/users/index.html<br>https://pandas.pydata.org/docs/user_guide/style.html<br>https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.style.html<br>https://pandas.pydata.org/docs/reference/style.html